# Purpose/risk on 20 labels: controller + model

The whole path on a small set: labels from the loader, prompts from the
purpose/risk spec, `QwenController.stream` driving `QwenModel`. Outcomes print
as each label settles, in completion order, then sit beside the gold reading.

Nothing is written to disk: rows stay in this notebook.

In [ ]:
import pandas as pd

from engine.controllers.qwen_controller import QwenController
from engine.models.base_model import Prompt
from engine.models.qwen_model import QwenModel
from engine.repositories.dsld.labels import loader
from engine.repositories.prompts import loader as prompts

## Is the GPU free?

The model needs ~21 GB. Anything listed under compute apps is holding the card.

In [2]:
!nvidia-smi --query-gpu=memory.free --format=csv
!nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv

memory.free [MiB]
24111 MiB
pid, process_name, used_gpu_memory [MiB]


## 20 labels and the prompt

The first 20 frozen pilot passages, in file order (the pilot file is already a
random draw). The spec brings the system prompt, the stop sequences, the output
budget and the checks the controller runs on every answer.

In [3]:
labels = loader.load(limit=20)
print(labels.summary())

spec = prompts.load("purpose_risk")
print(f"system prompt {len(spec.system)} chars, max_tokens {spec.max_tokens}, stop {spec.stop}")
print("checks:", [c.name for c in spec.checks])

20 to do of 1000 available
system prompt 6377 chars, max_tokens 400, stop ('<|im_end|>', '\nEND')
checks: ['line_prefix', 'line_score', 'claim_not_empty', 'not_truncated']


Work is a lazy generator of `(key, prompt)`: the controller pulls a label only
when it has room for one more in flight. The key is the DSLD id, and it comes
back with the outcome.

In [4]:
def work():
    for lab in labels:
        yield lab.dsld_id, Prompt(
            system=spec.system,
            user=lab.passage,
            stop=spec.stop,
            max_tokens=spec.max_tokens,
        )

## Start the model

~15 s with the weights cached, ~40 s cold. Refuses if the card is taken or the
model answers "Hello" in garbage.

In [5]:
model = QwenModel()
%time model.ready()

CPU times: user 23.5 ms, sys: 3.67 ms, total: 27.2 ms
Wall time: 22.4 s


## Run

`stream` caches the shared system turn in every slot from the first prompts,
keeps `in_flight` prompts outstanding (8, for 4 slots), checks each prompt
fits a slot before sending it, retries an answer that fails a check, and
yields each outcome as it settles.

In [6]:
controller = QwenController(model)
print(f"in_flight {controller.in_flight}, max_attempts {controller.max_attempts}")

outcomes = {}
for dsld_id, outcome in controller.stream(
    work(),
    spec.checks,
    total=len(labels),
    on_progress=lambda p: print(f"  [{p.done}/{p.total}] {p.detail}"),
):
    outcomes[dsld_id] = outcome
    first = outcome.generation.text.strip().splitlines()[:1]
    print(f"{dsld_id:>7}  ok={outcome.ok}  attempts={outcome.attempts}  {first[0] if first else '(no claims)'}")

in_flight 8, max_attempts 3
  [1/20] 1872 tok · 20.6 tok/s · eta 1m06s
    709  ok=True  attempts=1  P: supports memory retention | 5
  [2/20] 3952 tok · 37.7 tok/s · eta 31s
    885  ok=True  attempts=1  P: general health | 5
  [3/20] 5965 tok · 28.0 tok/s · eta 30s
   1690  ok=True  attempts=1  R: do not use if pregnant or nursing | 5
  [4/20] 8177 tok · 40.0 tok/s · eta 25s
    889  ok=True  attempts=1  P: goldenseal is traditionally used to support the health of the immune system | 5
  [5/20] 10365 tok · 42.2 tok/s · eta 22s
   1736  ok=True  attempts=1  R: not intended for use by persons under the age of 18 | 5
  [6/20] 13127 tok · 51.1 tok/s · eta 23s
   1024  ok=True  attempts=1  P: consumption of omega-3 fatty acids may reduce the risk of coronary heart disease | 5
  [7/20] 15910 tok · 56.0 tok/s · eta 26s
   2036  ok=True  attempts=1  P: formulated to help you burn fat and achieve your weight loss goals | 5
  [8/20] 19117 tok · 71.4 tok/s · eta 22s
   2417  ok=True  attempts=1

## Results

One row per label, in input order. `failed` names the checks the last attempt
broke; `fits_context` means the prompt was too long for a slot and never sent.

In [7]:
names = {lab.dsld_id: lab.name for lab in labels}
df = pd.DataFrame(
    {
        "dsld_id": i,
        "name": names[i],
        "ok": o.ok,
        "attempts": o.attempts,
        "failed": ", ".join(o.failed),
        "prompt_tok": o.generation.usage.prompt_tokens,
        "out_tok": o.generation.usage.completion_tokens,
        "latency_s": o.generation.latency_ms / 1000,
        "answer": o.generation.text.strip(),
    }
    for i, o in ((i, outcomes[i]) for i in labels.ids)
)
print(f"{df.ok.sum()}/{len(df)} ok, {(df.attempts > 1).sum()} needed a retry")
df

20/20 ok, 0 needed a retry


,dsld_id,name,ok,attempts,failed,prompt_tok,out_tok,latency_s,answer
0,709,Huperzine A,True,1,,1800,72,3.479,P: supports memory retention | 5\nR: do not ex...
1,885,Spirulina Blue-Green Algae 500 mg,True,1,,2020,60,3.474,P: general health | 5\nP: promotes general hea...
2,889,Goldenseal Root 470 mg,True,1,,2109,103,6.206,P: goldenseal is traditionally used to support...
3,1024,Fish Oil,True,1,,2583,179,9.611,P: consumption of omega-3 fatty acids may redu...
4,1690,White Willow Bark,True,1,,1999,14,1.723,R: do not use if pregnant or nursing | 5
5,1736,Ginkgo Leaf Extract 500 mg,True,1,,2123,65,3.943,R: not intended for use by persons under the a...
6,2036,Ultra Green Tea,True,1,,2482,301,8.954,P: formulated to help you burn fat and achieve...
7,2167,Ultralean,True,1,,2184,381,12.178,P: amplified metabolism | 5\nP: increased ener...
8,2417,Hydroxycut Advanced,True,1,,2940,267,7.429,P: clinically researched to lose up to 4 times...
9,2454,3-in-1 Joint Formula,True,1,,2686,245,7.133,P: rebuild | 5\nP: lubricate | 5\nP: soothe | ...


## Against the gold set

All 20 labels are in the gold set. Each answer printed beside the gold
reading's purpose and risk claims.

In [8]:
gold = loader.gold_by_id()

for i in labels.ids:
    g = gold.get(i)
    print(f"=== {i}  {names[i]}")
    print("model:")
    print("  " + (outcomes[i].generation.text.strip().replace("\n", "\n  ") or "(no claims)"))
    if g is None:
        print("gold: (not in gold set)")
    else:
        print("gold:")
        for c in g.purpose:
            print(f"  P: {c.text}")
        for c in g.risk:
            print(f"  R: {c.text}")
        if not g.purpose and not g.risk:
            print("  (no claims)")
    print()

=== 709  Huperzine A
model:
  P: supports memory retention | 5
  R: do not exceed recommended intake | 5
  R: consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition | 5
  R: discontinue use two weeks prior to surgery | 5
gold:
  P: supports memory retention
  R: do not exceed recommended intake
  R: consult a physician before use if pregnant, nursing, taking medication, or with high blood pressure, heart problems or a medical condition
  R: discontinue use two weeks prior to surgery

=== 885  Spirulina Blue-Green Algae 500 mg
model:
  P: general health | 5
  P: promotes general health and wellness | 5
  R: if you are pregnant, nursing or taking any medications, consult your doctor before use | 5
  R: discontinue use and consult your doctor if any adverse reactions occur | 5
gold:
  P: general health
  P: promotes general health and wellness
  R: consult a doctor before use 

## One label in full

Change `i` to look at another: the passage the model read, then what it said.

In [9]:
i = labels.ids[0]
print(next(lab for lab in labels if lab.dsld_id == i).passage)
print("\n---\n")
print(outcomes[i].generation.text)

PRODUCT: Huperzine A
BRAND: GNC Herbal Plus Standardized
TYPE: Botanical (Capsule)
CLAIM TYPES ON LABEL: All Other, Structure/Function
TARGET GROUPS: Vegetarian, Adult (18 - 50 Years)
INGREDIENTS: Huperzine A

LABEL STATEMENTS:
[Precautions re: All Other] KEEP OUT OF REACH OF CHILDREN.
[Precautions re: Pregnant or Nursing or Prescription Medications] WARNING: Do not exceed recommended intake. Consult your physicial prior to using this product if you are pregnant, nursing, taking medication, have high blood pressure, heart problems or have a medical condition. Discontinue use two weeks prior to surgery.
[Storage] Store in a cool, dry place.
[General Statements: All Other Content] Supports Memory Retention*
[Suggested/Recommended/Usage/Directions] Directions: As a dietary supplement, take one to two capsules daily.

---

P: supports memory retention | 5
R: do not exceed recommended intake | 5
R: consult your physicial prior to using this product if you are pregnant, nursing, taking medic

## Close

Stops the server and frees the GPU.

In [10]:
model.close()